# 04 — Exploratory analysis

Document hourly trends, weekday/weekend profiles, high-demand zones, sparsity and exploratory zone classes.

**Execution status:** Code is provided for reproducibility. See `results/observed_results.json` for results independently reported from an earlier Fabric run; this generated notebook has not been executed in this environment.

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
from pyspark.sql.window import Window
gold=spark.table("gold_hourly_zone_demand")
silver=spark.table("silver_yellow_taxi_trips")
eda=(gold.withColumn("pickup_date",F.to_date("pickup_hour"))
 .withColumn("hour_of_day",F.hour("pickup_hour"))
 .withColumn("day_type",F.when(F.dayofweek("pickup_hour").isin(1,7),"Weekend").otherwise("Weekday")))

In [ ]:
city_hourly=eda.groupBy("hour_of_day").agg(F.sum("pickup_count").alias("total_pickups")).orderBy("hour_of_day")
city_hourly.show(24)
daily_hourly=eda.groupBy("pickup_date","day_type","hour_of_day").agg(F.sum("pickup_count").alias("hourly_pickups"))
hourly_comparison=daily_hourly.groupBy("day_type","hour_of_day").agg(F.round(F.avg("hourly_pickups"),2).alias("avg_hourly_pickups"))
hourly_comparison.orderBy("hour_of_day","day_type").show(48)

In [ ]:
pd_plot=hourly_comparison.toPandas()
fig,ax=plt.subplots(figsize=(12,5))
for label in ("Weekday","Weekend"):
    subset=pd_plot[pd_plot.day_type==label].sort_values("hour_of_day")
    ax.plot(subset.hour_of_day,subset.avg_hourly_pickups,marker="o",label=label)
ax.set(title="NYC Yellow Taxi: Average Hourly Pickups by Day Type, Jan 2025",xlabel="Hour of day",ylabel="Average recorded pickups")
ax.set_xticks(range(24));ax.grid(alpha=.25);ax.legend();fig.tight_layout();plt.show()

In [ ]:
lookup=silver.select("PULocationID","pickup_borough","pickup_zone").dropDuplicates(["PULocationID"])
zone_summary=(gold.groupBy("PULocationID").agg(F.sum("pickup_count").alias("total_pickups"),F.avg("pickup_count").alias("avg_hourly_pickups"),F.sum(F.when(F.col("pickup_count")==0,1).otherwise(0)).alias("zero_hours"))
 .withColumn("zero_hour_pct",F.round(F.col("zero_hours")/744*100,2))
 .join(lookup,"PULocationID","left"))
zone_summary.orderBy(F.desc("total_pickups")).show(10,truncate=False)
zone_summary.orderBy(F.desc("zero_hour_pct")).show(10,truncate=False)

In [ ]:
exploratory_categories=(zone_summary.withColumn("demand_category",F.when(F.col("avg_hourly_pickups")<1,"Very Low").when(F.col("avg_hourly_pickups")<10,"Low").when(F.col("avg_hourly_pickups")<50,"Medium").otherwise("High")))
exploratory_categories.groupBy("demand_category").agg(F.count("*").alias("zones"),F.sum("total_pickups").alias("pickups")).show()

These categories use all January data **for EDA only**. Any predictive category must be fit from training history; see notebook 07. Results visible from the prior user Fabric run are transcribed in `results/observed_results.json`.